IMPORTING REQUIRED LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from xgboost import XGBRegressor

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

import warnings
warnings.filterwarnings("ignore")

DATA LOADING

In [ ]:
file_path = r"D:\MSC THESIS AAVAS\AFTER MID DEFENSE\FINAL DATA\Final Data for Model.xlsx"
df = pd.read_excel(file_path)

INITIAL INSPECTION OF DATA

In [15]:
df.head()

,SN,V85 Speed,Gradient,Radius of Curve,Carriageway Width,Left Shoulder Width,Left Shoulder Type,Right Shoulder Width,Right Shoulder Type,IRI
0,1,57.0,1.27,inf,5.5,0.9,Gravel,0.9,Gravel,3.70
1,2,33.0,2.92,46.0,5.5,0.9,Gravel,0.9,Gravel,4.51
2,3,43.0,7.97,inf,5.5,0.9,Gravel,0.9,Gravel,4.39
3,4,48.0,2.07,inf,5.5,0.9,Gravel,0.9,Gravel,4.69
4,5,36.0,1.21,16.0,5.5,0.9,Gravel,0.9,Gravel,5.57


In [16]:
df.describe()

,SN,V85 Speed,Gradient,Radius of Curve,Carriageway Width,Left Shoulder Width,Right Shoulder Width,IRI
count,165.000000,165.000000,165.000000,165.0,165.000000,165.000000,165.000000,165.000000
mean,89.145455,44.525758,4.100955,inf,6.323788,0.622606,0.628545,5.922667
std,50.827213,9.485801,2.498089,NaN,0.894131,0.407672,0.397181,2.636332
min,1.000000,25.000000,0.070000,15.0,4.700000,0.000000,0.000000,2.150000
25%,47.000000,37.000000,1.850000,35.0,5.500000,0.300000,0.380000,3.700000
50%,89.000000,44.000000,4.090000,75.0,7.000000,0.750000,0.750000,5.350000
75%,132.000000,50.000000,6.000000,NaN,7.000000,1.000000,1.000000,8.340000
max,175.000000,67.250000,11.085000,inf,8.000000,1.130000,1.180000,12.050000


In [17]:
df.shape

(165, 10)

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 165 entries, 0 to 164
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   SN                    165 non-null    int64  
 1   V85 Speed             165 non-null    float64
 2   Gradient              165 non-null    float64
 3   Radius of Curve       165 non-null    float64
 4   Carriageway Width     165 non-null    float64
 5   Left Shoulder Width   165 non-null    float64
 6   Left Shoulder Type    165 non-null    str    
 7   Right Shoulder Width  165 non-null    float64
 8   Right Shoulder Type   165 non-null    str    
 9   IRI                   165 non-null    float64
dtypes: float64(7), int64(1), str(2)
memory usage: 13.0 KB


DATA CLEANING AND PREPARATION

Missing Values Check

In [19]:
df.isnull().sum()

SN                      0
V85 Speed               0
Gradient                0
Radius of Curve         0
Carriageway Width       0
Left Shoulder Width     0
Left Shoulder Type      0
Right Shoulder Width    0
Right Shoulder Type     0
IRI                     0
dtype: int64

Cleaning Categorical Variables

In [20]:
df["Left Shoulder Type"] = df["Left Shoulder Type"].str.strip().str.lower().str.capitalize()
df["Right Shoulder Type"] = df["Right Shoulder Type"].str.strip().str.lower().str.capitalize()

Encoding

In [21]:
df_encoded = pd.get_dummies(
    df,
    columns=["Left Shoulder Type", "Right Shoulder Type"],
    drop_first=True,
    dtype=int
)

Variance Inflation Factor(VIF for Multicollinearity Check)

In [25]:
X_vif = df_encoded[
    [
        "Gradient",
        "Radius of Curve",
        "Carriageway Width",
        "Left Shoulder Width",
        "Right Shoulder Width",
        "Left Shoulder Type_Gravel",
        "Left Shoulder Type_Paved",
        "Right Shoulder Type_Gravel",
        "Right Shoulder Type_Paved"
    ]
]

X_vif = X_vif.astype(float)
X_vif_const = sm.add_constant(X_vif)

vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif_const.columns
vif_data["VIF"] = [
    variance_inflation_factor(X_vif_const.values, i)
    for i in range(X_vif_const.shape[1])
]

vif_data

MissingDataError: exog contains inf or nans

Final Features Selection

In [ ]:
X = df_encoded[
    [
        "Gradient",
        "Radius of Curve",
        "Carriageway Width",
        "Left Shoulder Width",
        "Right Shoulder Width",
        "Left Shoulder Type_Gravel",
        "Left Shoulder Type_Paved",
        "Right Shoulder Type_Gravel",
        "Right Shoulder Type_Paved"
    ]
]

y = df_encoded["V85 Speed"]

EXPLORATORY DATA ANALYSIS

Distribution Plots

In [ ]:
num_cols = [
    "V85 Speed",
    "Gradient",
    "Carriageway Width",
    "Left Shoulder Width",
    "Right Shoulder Width",
    "Radius of Curve"
]

sns.set_style("white")

for col in num_cols:
    plt.figure(figsize=(7,4))

    sns.histplot(df[col], bins=20, stat="density", color="skyblue", edgecolor="black")
    sns.kdeplot(df[col], color="red", linewidth=2)

    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Density")

    plt.tight_layout()
    plt.show()



Boxplots

In [ ]:
sns.set_style("whitegrid")

for col in num_cols:
    plt.figure(figsize=(7,4))

    sns.boxplot(x=df[col], color="lightblue", width=0.4)

    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)

    plt.tight_layout()
    plt.show()

Categorical Variable Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 🔹 1. Left Shoulder Type Count
sns.countplot(
    x=df["Left Shoulder Type"],
    palette="pastel",
    ax=axes[0, 0]
)
axes[0, 0].set_title("Distribution of Left Shoulder Type", fontweight="bold")
axes[0, 0].set_xlabel("Left Shoulder Type")
axes[0, 0].set_ylabel("Count")
axes[0, 0].tick_params(axis='x', rotation=45)

# 🔹 2. Right Shoulder Type Count
sns.countplot(
    x=df["Right Shoulder Type"],
    palette="pastel",
    ax=axes[0, 1]
)
axes[0, 1].set_title("Distribution of Right Shoulder Type", fontweight="bold")
axes[0, 1].set_xlabel("Right Shoulder Type")
axes[0, 1].set_ylabel("Count")
axes[0, 1].tick_params(axis='x', rotation=45)

# 🔹 3. Left Shoulder Type vs Speed
sns.boxplot(
    x=df["Left Shoulder Type"],
    y=df["V85 Speed"],
    palette="pastel",
    ax=axes[1, 0]
)
axes[1, 0].set_title("Left Shoulder Type vs V85 Speed", fontweight="bold")
axes[1, 0].set_xlabel("Left Shoulder Type")
axes[1, 0].set_ylabel("V85 Speed")
axes[1, 0].tick_params(axis='x', rotation=45)

# 🔹 4. Right Shoulder Type vs Speed
sns.boxplot(
    x=df["Right Shoulder Type"],
    y=df["V85 Speed"],
    palette="pastel",
    ax=axes[1, 1]
)
axes[1, 1].set_title("Right Shoulder Type vs V85 Speed", fontweight="bold")
axes[1, 1].set_xlabel("Right Shoulder Type")
axes[1, 1].set_ylabel("V85 Speed")
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("Shoulder_Type_Analysis.png", dpi=300)
plt.show()

Skewness

In [ ]:
skew_table = pd.DataFrame({
    "Variable": num_cols,
    "Skewness": df[num_cols].skew().values
})

skew_table

Correlation Matrix

In [ ]:
corr_matrix = df[num_cols].corr()
corr_matrix

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap="RdYlGn",        # your preferred colors
    vmin=-1, vmax=1,
    center=0,
    fmt=".2f",
    linewidths=0.8,
    linecolor="black",    # darker grid lines
    cbar_kws={"shrink": 0.8}
)

plt.title("Correlation Matrix of Variables", fontsize=14, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

Relationship Plots

In [ ]:
predictor_cols = [
    "Radius of Curve",
    "Gradient",
    "Carriageway Width",
    "Left Shoulder Width",
    "Right Shoulder Width"
]

In [ ]:
sns.set_style("white")

for col in predictor_cols:
    plt.figure(figsize=(7, 4))

    sns.regplot(
        x=df[col],
        y=df["V85 Speed"],
        scatter_kws={"s": 40, "color": "steelblue", "alpha": 0.7},
        line_kws={"color": "darkred", "linewidth": 2},
        ci=None
    )

    plt.title(f"{col} vs V85 Speed")
    plt.xlabel(col)
    plt.ylabel("V85 Speed")

    plt.tight_layout()
    plt.show()

MODEL DEVELOPMENT

Train-Test-Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)

Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

MODELING

MLR Model

Training of MLR Model

In [ ]:
from sklearn.linear_model import LinearRegression

mlr = LinearRegression()
mlr.fit(X_train_scaled, y_train)

Predictions using MLR

In [ ]:
y_train_pred_mlr = mlr.predict(X_train_scaled)
y_test_pred_mlr = mlr.predict(X_test_scaled)

Model Evaluation Metrices

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np
import pandas as pd

mlr_train_r2 = r2_score(y_train, y_train_pred_mlr)
mlr_test_r2 = r2_score(y_test, y_test_pred_mlr)

mlr_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_mlr))
mlr_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_mlr))

mlr_train_mae = mean_absolute_error(y_train, y_train_pred_mlr)
mlr_test_mae = mean_absolute_error(y_test, y_test_pred_mlr)

mlr_results = pd.DataFrame({
    "Metric": ["R2", "RMSE", "MAE"],
    "Value": [mlr_test_r2, mlr_test_rmse, mlr_test_mae]
})

mlr_results

Table of Coefficient of Variables

In [ ]:
coef_table_mlr = pd.DataFrame({
    "Variable": X.columns,
    "Coefficient": mlr.coef_
})

# Add intercept as first row
intercept_row = pd.DataFrame({
    "Variable": ["Constant"],
    "Coefficient": [mlr.intercept_]
})

coef_table_mlr = pd.concat([intercept_row, coef_table_mlr], ignore_index=True)

coef_table_mlr["Coefficient"] = coef_table_mlr["Coefficient"].round(4)

coef_table_mlr

Actual vs Predicted Plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

# Scatter plot
sns.scatterplot(
    x=y_test,
    y=y_test_pred_mlr,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

# Perfect fit line
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="darkred",
    linestyle="--",
    linewidth=2,
    label="Perfect Fit"
)

plt.xlabel("Actual V85 Speed")
plt.ylabel("Predicted V85 Speed")
plt.title("MLR: Actual vs Predicted", fontsize=12, fontweight="bold")

plt.legend()
plt.tight_layout()
plt.show()

Residual Plot

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")

# Calculate residuals
mlr_residuals = y_test - y_test_pred_mlr

plt.figure(figsize=(7, 4))

sns.scatterplot(
    x=y_test_pred_mlr,
    y=mlr_residuals,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

plt.axhline(y=0, linestyle="--", color="darkred", linewidth=2)

plt.xlabel("Predicted V85 Speed")
plt.ylabel("Residuals")
plt.title("MLR: Residual Plot", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

Result Table

In [ ]:
mlr_final = pd.DataFrame({
    "Model": ["MLR"],
    "R2": [round(mlr_test_r2, 3)],
    "RMSE": [round(mlr_test_rmse, 3)],
    "MAE": [round(mlr_test_mae, 3)]
})

mlr_final

Random Forest Model

Random Forest Model Training

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    random_state=42
)

rf.fit(X_train_scaled, y_train)

Predictions Using Random Forest Model

In [ ]:
y_train_pred_rf = rf.predict(X_train_scaled)
y_test_pred_rf = rf.predict(X_test_scaled)

Model Evaluation Metrices

In [ ]:
rf_train_r2 = r2_score(y_train, y_train_pred_rf)
rf_test_r2 = r2_score(y_test, y_test_pred_rf)

rf_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_rf))
rf_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_rf))

rf_train_mae = mean_absolute_error(y_train, y_train_pred_rf)
rf_test_mae = mean_absolute_error(y_test, y_test_pred_rf)

rf_results = pd.DataFrame({
    "Metric": ["R2", "RMSE", "MAE"],
    "Value": [rf_test_r2, rf_test_rmse, rf_test_mae]
})

rf_results

Feature Importance Plot and Table

In [ ]:
rf_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

rf_importance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.barplot(
    x="Importance",
    y="Feature",
    data=rf_importance,  # make sure this matches your dataframe
    palette="pastel"
)

plt.title("Feature Importance (Random Forest)", fontsize=10, fontweight="bold")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

Actual vs Predicted Plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

# Scatter plot using seaborn
sns.scatterplot(
    x=y_test,
    y=y_test_pred_rf,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

# Perfect fit line (45-degree line)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="darkred",
    linestyle="--",
    linewidth=2,
    label="Perfect Fit"
)

plt.xlabel("Actual V85 Speed")
plt.ylabel("Predicted V85 Speed")
plt.title("Random Forest: Actual vs Predicted", fontsize=12, fontweight="bold")

plt.legend()
plt.tight_layout()
plt.show()

Residual Plot

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")

rf_residuals = y_test - y_test_pred_rf

plt.figure(figsize=(7, 4))

sns.scatterplot(
    x=y_test_pred_rf,
    y=rf_residuals,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

plt.axhline(y=0, linestyle="--", color="darkred", linewidth=2)

plt.xlabel("Predicted V85 Speed")
plt.ylabel("Residuals")
plt.title("Random Forest: Residual Plot", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

Result Table

In [ ]:
rf_final = pd.DataFrame({
    "Model": ["Random Forest"],
    "R2": [round(rf_test_r2, 3)],
    "RMSE": [round(rf_test_rmse, 3)],
    "MAE": [round(rf_test_mae, 3)]
})

rf_final

Hyperparameter Tuned Random Forest Model

Hyperparameter Grid for Random Forest

In [ ]:
param_grid_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

Grid Search Setup and Training

In [ ]:
from sklearn.model_selection import GridSearchCV

grid_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid_rf,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_train_scaled, y_train)

print("Tuning completed")
print("Best Parameters:", grid_rf.best_params_)
print("Best CV Score:", round(grid_rf.best_score_, 3))

Extracting Best Tuned Random Forest Model

In [ ]:
best_rf = grid_rf.best_estimator_
best_rf

Predictions Using Tuned Random Forest Model

In [ ]:
y_train_pred_rf_tuned = best_rf.predict(X_train_scaled)
y_test_pred_rf_tuned = best_rf.predict(X_test_scaled)

Model Evaluation Metrices

In [ ]:
rf_tuned_train_r2 = r2_score(y_train, y_train_pred_rf_tuned)
rf_tuned_test_r2 = r2_score(y_test, y_test_pred_rf_tuned)

rf_tuned_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_rf_tuned))
rf_tuned_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_rf_tuned))

rf_tuned_train_mae = mean_absolute_error(y_train, y_train_pred_rf_tuned)
rf_tuned_test_mae = mean_absolute_error(y_test, y_test_pred_rf_tuned)

rf_tuned_results = pd.DataFrame({
    "Metric": ["R2", "RMSE", "MAE"],
    "Value": [rf_tuned_test_r2, rf_tuned_test_rmse, rf_tuned_test_mae]
})

rf_tuned_results

Feature Importance Table and PLot

In [ ]:
rf_tuned_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

rf_tuned_importance

In [ ]:
sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.barplot(
    x="Importance",
    y="Feature",
    data=rf_tuned_importance,
    palette="pastel"
)

plt.title("Feature Importance (Tuned Random Forest)", fontsize=10, fontweight="bold")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

Actual vs Predicted Plot

In [ ]:
sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.scatterplot(
    x=y_test,
    y=y_test_pred_rf_tuned,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="darkred",
    linestyle="--",
    linewidth=2,
    label="Perfect Fit"
)

plt.xlabel("Actual V85 Speed")
plt.ylabel("Predicted V85 Speed")
plt.title("Tuned Random Forest: Actual vs Predicted", fontsize=12, fontweight="bold")

plt.legend()
plt.tight_layout()
plt.show()

Residual Plot

In [ ]:
rf_tuned_residuals = y_test - y_test_pred_rf_tuned

sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.scatterplot(
    x=y_test_pred_rf_tuned,
    y=rf_tuned_residuals,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

plt.axhline(y=0, linestyle="--", color="darkred", linewidth=2)

plt.xlabel("Predicted V85 Speed")
plt.ylabel("Residuals")
plt.title("Tuned Random Forest: Residual Plot", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

Result Table

In [ ]:
rf_tuned_final = pd.DataFrame({
    "Model": ["Tuned Random Forest"],
    "R2": [round(rf_tuned_test_r2, 3)],
    "RMSE": [round(rf_tuned_test_rmse, 3)],
    "MAE": [round(rf_tuned_test_mae, 3)]
})

rf_tuned_final

Base vs Tuned Comparison Table

In [ ]:
rf_comparison = pd.concat([rf_final, rf_tuned_final], ignore_index=True)
rf_comparison

Best Hyperparameters Table

In [ ]:
best_rf_params = pd.DataFrame(
    list(grid_rf.best_params_.items()),
    columns=["Parameter", "Best Value"]
)

best_rf_params

XGBoost Model

Model Training

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    random_state=42,
    objective="reg:squarederror"
)

xgb.fit(X_train_scaled, y_train)

Model Predictions

In [23]:
y_train_pred_xgb = xgb.predict(X_train_scaled)
y_test_pred_xgb = xgb.predict(X_test_scaled)

NameError: name 'xgb' is not defined

Model Evaluation Metrices

In [ ]:
xgb_train_r2 = r2_score(y_train, y_train_pred_xgb)
xgb_test_r2 = r2_score(y_test, y_test_pred_xgb)

xgb_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_xgb))
xgb_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_xgb))

xgb_train_mae = mean_absolute_error(y_train, y_train_pred_xgb)
xgb_test_mae = mean_absolute_error(y_test, y_test_pred_xgb)

xgb_results = pd.DataFrame({
    "Metric": ["R2", "RMSE", "MAE"],
    "Value": [xgb_test_r2, xgb_test_rmse, xgb_test_mae]
})

xgb_results

Feature Importance Table and Plot

In [ ]:
xgb_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb.feature_importances_
}).sort_values(by="Importance", ascending=False)

xgb_importance

In [ ]:
sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.barplot(
    x="Importance",
    y="Feature",
    data=xgb_importance,
    palette="pastel"
)

plt.title("Feature Importance (XGBoost)", fontsize=10, fontweight="bold")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

Actual vs Predicted Plot

In [ ]:
sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.scatterplot(
    x=y_test,
    y=y_test_pred_xgb,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="darkred",
    linestyle="--",
    linewidth=2,
    label="Perfect Fit"
)

plt.xlabel("Actual V85 Speed")
plt.ylabel("Predicted V85 Speed")
plt.title("XGBoost: Actual vs Predicted", fontsize=12, fontweight="bold")

plt.legend()
plt.tight_layout()
plt.show()

Residual Plot

In [ ]:
xgb_residuals = y_test - y_test_pred_xgb

sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))
sns.scatterplot(
    x=y_test_pred_xgb,
    y=xgb_residuals,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

plt.axhline(y=0, linestyle="--", color="darkred", linewidth=2)

plt.xlabel("Predicted V85 Speed")
plt.ylabel("Residuals")
plt.title("XGBoost: Residual Plot", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

Result Table

In [ ]:
xgb_final = pd.DataFrame({
    "Model": ["XGBoost"],
    "R2": [round(xgb_test_r2, 3)],
    "RMSE": [round(xgb_test_rmse, 3)],
    "MAE": [round(xgb_test_mae, 3)]
})

xgb_final

Hyperparameter Tuned XGBoost Model

Hyperparameter Grid Definition

In [ ]:
param_grid_xgb = {
    "n_estimators": [100, 200],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0]
}

Grid Search Setup and Training

In [ ]:
grid_xgb = GridSearchCV(
    estimator=XGBRegressor(
        random_state=42,
        objective="reg:squarederror"
    ),
    param_grid=param_grid_xgb,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid_xgb.fit(X_train_scaled, y_train)

print("Best Parameters:", grid_xgb.best_params_)
print("Best CV Score:", round(grid_xgb.best_score_, 3))

Best Model Extraction

In [ ]:
best_xgb = grid_xgb.best_estimator_

Predictions Using Tuned Model

In [ ]:
y_train_pred_xgb_tuned = best_xgb.predict(X_train_scaled)
y_test_pred_xgb_tuned = best_xgb.predict(X_test_scaled)

Model Evaluation Metrices

In [ ]:
xgb_tuned_train_r2 = r2_score(y_train, y_train_pred_xgb_tuned)
xgb_tuned_test_r2 = r2_score(y_test, y_test_pred_xgb_tuned)

xgb_tuned_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_xgb_tuned))
xgb_tuned_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_xgb_tuned))

xgb_tuned_train_mae = mean_absolute_error(y_train, y_train_pred_xgb_tuned)
xgb_tuned_test_mae = mean_absolute_error(y_test, y_test_pred_xgb_tuned)

xgb_tuned_results = pd.DataFrame({
    "Metric": ["R2", "RMSE", "MAE"],
    "Value": [xgb_tuned_test_r2, xgb_tuned_test_rmse, xgb_tuned_test_mae]
})

xgb_tuned_results

Feature Importance Table and Plot

In [ ]:
xgb_tuned_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_xgb.feature_importances_
})

# Sort from highest to lowest importance
xgb_tuned_importance = xgb_tuned_importance.sort_values(
    by="Importance",
    ascending=False
).reset_index(drop=True)

# Round values for better presentation
xgb_tuned_importance["Importance"] = xgb_tuned_importance["Importance"].round(4)

xgb_tuned_importance

In [ ]:
sns.set_style("whitegrid")

# Sort properly (important)
xgb_tuned_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_xgb.feature_importances_
}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(7, 4))

sns.barplot(
    x="Importance",
    y="Feature",
    data=xgb_tuned_importance,
    palette="pastel"
)

plt.title("Feature Importance (Tuned XGBoost)", fontsize=10, fontweight="bold")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

Actual vs Predicted Plot

In [ ]:
sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.scatterplot(
    x=y_test,
    y=y_test_pred_xgb_tuned,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

# Perfect fit line
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="darkred",
    linestyle="--",
    linewidth=2,
    label="Perfect Fit"
)

plt.xlabel("Actual V85 Speed")
plt.ylabel("Predicted V85 Speed")
plt.title("Tuned XGBoost: Actual vs Predicted", fontsize=12, fontweight="bold")

plt.legend()
plt.tight_layout()
plt.show()

Residual Plot

In [ ]:
xgb_tuned_residuals = y_test - y_test_pred_xgb_tuned

sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.scatterplot(
    x=y_test_pred_xgb_tuned,
    y=xgb_tuned_residuals,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

plt.axhline(y=0, linestyle="--", color="darkred", linewidth=2)

plt.xlabel("Predicted V85 Speed")
plt.ylabel("Residuals")
plt.title("Tuned XGBoost: Residual Plot", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

Result Table

In [ ]:
xgb_tuned_final = pd.DataFrame({
    "Model": ["Tuned XGBoost"],
    "R2": [round(xgb_tuned_test_r2, 3)],
    "RMSE": [round(xgb_tuned_test_rmse, 3)],
    "MAE": [round(xgb_tuned_test_mae, 3)]
})

xgb_tuned_final

Base vs Tuned Model Comparison

In [ ]:
xgb_comparison = pd.concat([xgb_final, xgb_tuned_final], ignore_index=True)
xgb_comparison

Best Hyperparameters Table

In [ ]:
best_xgb_params = pd.DataFrame(
    list(grid_xgb.best_params_.items()),
    columns=["Parameter", "Best Value"]
)

best_xgb_params

Support Vector Regression (SVR) Model

Model Training

In [ ]:
from sklearn.svm import SVR

svr = SVR(
    kernel="rbf",
    C=100,
    epsilon=0.1,
    gamma="scale"
)

svr.fit(X_train_scaled, y_train)

Model Predictions

In [ ]:
y_train_pred_svr = svr.predict(X_train_scaled)
y_test_pred_svr = svr.predict(X_test_scaled)

Model Evaluation Metrices

In [ ]:
svr_train_r2 = r2_score(y_train, y_train_pred_svr)
svr_test_r2 = r2_score(y_test, y_test_pred_svr)

svr_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_svr))
svr_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_svr))

svr_train_mae = mean_absolute_error(y_train, y_train_pred_svr)
svr_test_mae = mean_absolute_error(y_test, y_test_pred_svr)

svr_results = pd.DataFrame({
    "Metric": ["R2", "RMSE", "MAE"],
    "Value": [svr_test_r2, svr_test_rmse, svr_test_mae]
})

svr_results

Actual vs Predicted Plot


In [ ]:
sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.scatterplot(
    x=y_test,
    y=y_test_pred_svr,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="darkred",
    linestyle="--",
    linewidth=2,
    label="Perfect Fit"
)

plt.xlabel("Actual V85 Speed")
plt.ylabel("Predicted V85 Speed")
plt.title("SVR: Actual vs Predicted", fontsize=12, fontweight="bold")

plt.legend()
plt.tight_layout()
plt.show()

Residual Plot

In [ ]:
svr_residuals = y_test - y_test_pred_svr

sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.scatterplot(
    x=y_test_pred_svr,
    y=svr_residuals,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

plt.axhline(y=0, linestyle="--", color="darkred", linewidth=2)

plt.xlabel("Predicted V85 Speed")
plt.ylabel("Residuals")
plt.title("SVR: Residual Plot", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

Result Table

In [ ]:
svr_final = pd.DataFrame({
    "Model": ["SVR"],
    "R2": [round(svr_test_r2, 3)],
    "RMSE": [round(svr_test_rmse, 3)],
    "MAE": [round(svr_test_mae, 3)]
})

svr_final

Hyperparameter Tuned SVR Model

Hyperparameter Grid Definition

In [ ]:
param_grid_svr = {
    "C": [1, 10, 100, 1000],
    "epsilon": [0.01, 0.1, 0.5, 1],
    "gamma": ["scale", 0.01, 0.1, 1],
    "kernel": ["rbf"]
}

Grid Search Setup and Training

In [ ]:
grid_svr = GridSearchCV(
    estimator=SVR(),
    param_grid=param_grid_svr,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid_svr.fit(X_train_scaled, y_train)

print("Best Parameters:", grid_svr.best_params_)
print("Best CV Score:", round(grid_svr.best_score_, 3))

Best Model Extraction

In [ ]:
best_svr = grid_svr.best_estimator_

Predictions Using Tuned Model

In [ ]:
y_train_pred_svr_tuned = best_svr.predict(X_train_scaled)
y_test_pred_svr_tuned = best_svr.predict(X_test_scaled)

Model Evaluation Metrices

In [ ]:
svr_tuned_train_r2 = r2_score(y_train, y_train_pred_svr_tuned)
svr_tuned_test_r2 = r2_score(y_test, y_test_pred_svr_tuned)

svr_tuned_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_svr_tuned))
svr_tuned_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_svr_tuned))

svr_tuned_train_mae = mean_absolute_error(y_train, y_train_pred_svr_tuned)
svr_tuned_test_mae = mean_absolute_error(y_test, y_test_pred_svr_tuned)

svr_tuned_results = pd.DataFrame({
    "Metric": ["R2", "RMSE", "MAE"],
    "Value": [svr_tuned_test_r2, svr_tuned_test_rmse, svr_tuned_test_mae]
})

svr_tuned_results

Actual vs Predicted Plot

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.scatterplot(
    x=y_test,
    y=y_test_pred_svr_tuned,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

# Perfect fit line
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="darkred",
    linestyle="--",
    linewidth=2,
    label="Perfect Fit"
)

plt.xlabel("Actual V85 Speed")
plt.ylabel("Predicted V85 Speed")
plt.title("Tuned SVR: Actual vs Predicted", fontsize=12, fontweight="bold")

plt.legend()
plt.tight_layout()
plt.show()

Residual Plot

In [ ]:
svr_tuned_residuals = y_test - y_test_pred_svr_tuned

sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.scatterplot(
    x=y_test_pred_svr_tuned,
    y=svr_tuned_residuals,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

plt.axhline(y=0, linestyle="--", color="darkred", linewidth=2)

plt.xlabel("Predicted V85 Speed")
plt.ylabel("Residuals")
plt.title("Tuned SVR: Residual Plot", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

Result Table

In [ ]:
svr_tuned_final = pd.DataFrame({
    "Model": ["Tuned SVR"],
    "R2": [round(svr_tuned_test_r2, 3)],
    "RMSE": [round(svr_tuned_test_rmse, 3)],
    "MAE": [round(svr_tuned_test_mae, 3)]
})

svr_tuned_final

Base vs Tuned Comparison Table

In [ ]:
svr_comparison = pd.concat([svr_final, svr_tuned_final], ignore_index=True)
svr_comparison

Best Hyperparameters Table

In [ ]:
best_svr_params = pd.DataFrame(
    list(grid_svr.best_params_.items()),
    columns=["Parameter", "Best Value"]
)

best_svr_params

Artificial Neural Network (ANN) Model

Model Training

In [ ]:
from sklearn.neural_network import MLPRegressor

ann = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    alpha=0.001,
    learning_rate_init=0.01,
    max_iter=2000,
    random_state=42
)

ann.fit(X_train_scaled, y_train)

Model Predictions

In [ ]:
y_train_pred_ann = ann.predict(X_train_scaled)
y_test_pred_ann = ann.predict(X_test_scaled)

Model Evaluation Metrices

In [ ]:
ann_train_r2 = r2_score(y_train, y_train_pred_ann)
ann_test_r2 = r2_score(y_test, y_test_pred_ann)

ann_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_ann))
ann_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_ann))

ann_train_mae = mean_absolute_error(y_train, y_train_pred_ann)
ann_test_mae = mean_absolute_error(y_test, y_test_pred_ann)

ann_results = pd.DataFrame({
    "Metric": ["R2", "RMSE", "MAE"],
    "Value": [ann_test_r2, ann_test_rmse, ann_test_mae]
})

ann_results

Actual vs Predicted Plot

In [ ]:
sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.scatterplot(
    x=y_test,
    y=y_test_pred_ann,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="darkred",
    linestyle="--",
    linewidth=2,
    label="Perfect Fit"
)

plt.xlabel("Actual V85 Speed")
plt.ylabel("Predicted V85 Speed")
plt.title("ANN: Actual vs Predicted", fontsize=12, fontweight="bold")

plt.legend()
plt.tight_layout()
plt.show()

Residual PLot

In [ ]:
ann_residuals = y_test - y_test_pred_ann

sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.scatterplot(
    x=y_test_pred_ann,
    y=ann_residuals,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

plt.axhline(y=0, linestyle="--", color="darkred", linewidth=2)

plt.xlabel("Predicted V85 Speed")
plt.ylabel("Residuals")
plt.title("ANN: Residual Plot", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

Result Table

In [ ]:
ann_final = pd.DataFrame({
    "Model": ["ANN"],
    "R2": [round(ann_test_r2, 3)],
    "RMSE": [round(ann_test_rmse, 3)],
    "MAE": [round(ann_test_mae, 3)]
})

ann_final

Hyperparameter Tuned ANN Model

Hyperparameter Grid Definition

In [ ]:
param_grid_ann = {
    "hidden_layer_sizes": [(32,), (64,), (64, 32), (128, 64)],
    "activation": ["relu", "tanh"],
    "alpha": [0.0001, 0.001, 0.01],
    "learning_rate_init": [0.001, 0.01]
}

Grid Search Setup and Training

In [ ]:
grid_ann = GridSearchCV(
    estimator=MLPRegressor(
        max_iter=3000,
        random_state=42
    ),
    param_grid=param_grid_ann,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid_ann.fit(X_train_scaled, y_train)

print("Best Parameters:", grid_ann.best_params_)
print("Best CV Score:", round(grid_ann.best_score_, 3))

Best Model Extraction

In [ ]:
best_ann = grid_ann.best_estimator_

Prediction Using Tuned Model

In [24]:
y_train_pred_ann_tuned = best_ann.predict(X_train_scaled)
y_test_pred_ann_tuned = best_ann.predict(X_test_scaled)

NameError: name 'best_ann' is not defined

Model Evaluation Metrices

In [ ]:
ann_tuned_train_r2 = r2_score(y_train, y_train_pred_ann_tuned)
ann_tuned_test_r2 = r2_score(y_test, y_test_pred_ann_tuned)

ann_tuned_train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred_ann_tuned))
ann_tuned_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_ann_tuned))

ann_tuned_train_mae = mean_absolute_error(y_train, y_train_pred_ann_tuned)
ann_tuned_test_mae = mean_absolute_error(y_test, y_test_pred_ann_tuned)

ann_tuned_results = pd.DataFrame({
    "Metric": ["R2", "RMSE", "MAE"],
    "Value": [ann_tuned_test_r2, ann_tuned_test_rmse, ann_tuned_test_mae]
})

ann_tuned_results

Actual vs Predicted Plot

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.scatterplot(
    x=y_test,
    y=y_test_pred_ann_tuned,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

# Perfect fit line
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="darkred",
    linestyle="--",
    linewidth=2,
    label="Perfect Fit"
)

plt.xlabel("Actual V85 Speed")
plt.ylabel("Predicted V85 Speed")
plt.title("Tuned ANN: Actual vs Predicted", fontsize=12, fontweight="bold")

plt.legend()
plt.tight_layout()
plt.show()

Residual Plot

In [ ]:
ann_tuned_residuals = y_test - y_test_pred_ann_tuned

sns.set_style("whitegrid")

plt.figure(figsize=(7, 4))

sns.scatterplot(
    x=y_test_pred_ann_tuned,
    y=ann_tuned_residuals,
    color="steelblue",
    s=60,
    alpha=0.7,
    edgecolor="black"
)

plt.axhline(y=0, linestyle="--", color="darkred", linewidth=2)

plt.xlabel("Predicted V85 Speed")
plt.ylabel("Residuals")
plt.title("Tuned ANN: Residual Plot", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

Result Table

In [ ]:
ann_tuned_final = pd.DataFrame({
    "Model": ["Tuned ANN"],
    "R2": [round(ann_tuned_test_r2, 3)],
    "RMSE": [round(ann_tuned_test_rmse, 3)],
    "MAE": [round(ann_tuned_test_mae, 3)]
})

ann_tuned_final

Base vs Tuned Comparison

In [ ]:
ann_comparison = pd.concat([ann_final, ann_tuned_final], ignore_index=True)
ann_comparison

Best Hyperparameters Table

In [ ]:
best_ann_params = pd.DataFrame(
    list(grid_ann.best_params_.items()),
    columns=["Parameter", "Best Value"]
)

best_ann_params

Final Comparison of All Models

In [ ]:

# Step 1: Add Type column properly
mlr_final["Type"] = "Base"

svr_final["Type"] = "Base"
svr_tuned_final["Type"] = "Tuned"

ann_final["Type"] = "Base"
ann_tuned_final["Type"] = "Tuned"

xgb_final["Type"] = "Base"
xgb_tuned_final["Type"] = "Tuned"

rf_final["Type"] = "Base"
rf_tuned_final["Type"] = "Tuned"

# Step 2: Combine all models
master_table = pd.concat([
    mlr_final,
    svr_final,
    svr_tuned_final,
    ann_final,
    ann_tuned_final,
    xgb_final,
    xgb_tuned_final,
    rf_final,
    rf_tuned_final
], ignore_index=True)

# Step 3: Arrange columns
master_table = master_table[["Model", "Type", "R2", "RMSE", "MAE"]]

# Step 4: Round values
master_table["R2"] = master_table["R2"].round(3)
master_table["RMSE"] = master_table["RMSE"].round(3)
master_table["MAE"] = master_table["MAE"].round(3)

# Step 5: Sort by RMSE (LOWEST = BEST)
master_table = master_table.sort_values(
    by="RMSE",
    ascending=True
).reset_index(drop=True)

# Step 6: Add ranking
master_table["Rank (RMSE)"] = range(1, len(master_table) + 1)

# Step 7: Final column order
master_table = master_table[["Rank (RMSE)", "Model", "Type", "R2", "RMSE", "MAE"]]

master_table

BOOTSTRAPPING for BEST TWO MODELS

Preparation of Data

In [ ]:
X_boot = X.copy()
y_boot = y.copy()

print(X_boot.shape, y_boot.shape)

Import Required Libraries

In [ ]:
from sklearn.utils import resample
from sklearn.base import clone

Creating Bootstrap Function

In [ ]:
def bootstrap_oob_evaluation(model, X, y, n_bootstrap=1000, random_state=42):
    np.random.seed(random_state)

    r2_list = []
    rmse_list = []
    mae_list = []
    used_iterations = []

    X_array = np.array(X)
    y_array = np.array(y)

    n = len(X_array)
    indices = np.arange(n)

    for i in range(n_bootstrap):
        # Bootstrap sample indices
        boot_idx = resample(indices, replace=True, n_samples=n, random_state=random_state + i)

        # Out-of-bag indices
        oob_idx = np.setdiff1d(indices, boot_idx)

        # Skip if no OOB sample
        if len(oob_idx) == 0:
            continue

        X_train_boot = X_array[boot_idx]
        y_train_boot = y_array[boot_idx]

        X_test_oob = X_array[oob_idx]
        y_test_oob = y_array[oob_idx]

        # Clone model so each iteration is fresh
        model_i = clone(model)
        model_i.fit(X_train_boot, y_train_boot)

        y_pred_oob = model_i.predict(X_test_oob)

        r2 = r2_score(y_test_oob, y_pred_oob)
        rmse = np.sqrt(mean_squared_error(y_test_oob, y_pred_oob))
        mae = mean_absolute_error(y_test_oob, y_pred_oob)

        r2_list.append(r2)
        rmse_list.append(rmse)
        mae_list.append(mae)
        used_iterations.append(i)

    results = pd.DataFrame({
        "Iteration": used_iterations,
        "R2": r2_list,
        "RMSE": rmse_list,
        "MAE": mae_list
    })

    return results

Bootstrap for Tuned RF

In [ ]:
rf_boot_results = bootstrap_oob_evaluation(
    model=best_rf,
    X=X_boot,
    y=y_boot,
    n_bootstrap=1000,
    random_state=42
)

rf_boot_results.head()

Bootstrap for Tuned XGBoost

In [ ]:
xgb_boot_results = bootstrap_oob_evaluation(
    model=best_xgb,
    X=X_boot,
    y=y_boot,
    n_bootstrap=1000,
    random_state=42
)

xgb_boot_results.head()

Summary Statistics

In [ ]:
def summarize_bootstrap_results(results, model_name):
    summary = pd.DataFrame({
        "Model": [model_name],
        "Mean R2": [results["R2"].mean()],
        "Std R2": [results["R2"].std()],
        "Mean RMSE": [results["RMSE"].mean()],
        "Std RMSE": [results["RMSE"].std()],
        "Mean MAE": [results["MAE"].mean()],
        "Std MAE": [results["MAE"].std()]
    })

    return summary.round(3)

rf_boot_summary = summarize_bootstrap_results(rf_boot_results, "Tuned Random Forest")
xgb_boot_summary = summarize_bootstrap_results(xgb_boot_results, "Tuned XGBoost")

boot_summary = pd.concat([rf_boot_summary, xgb_boot_summary], ignore_index=True)
boot_summary

95% Confidence Intervals

In [ ]:
def bootstrap_confidence_intervals(results, model_name):
    ci_summary = pd.DataFrame({
        "Model": [model_name],

        "R2 Mean": [results["R2"].mean()],
        "R2 2.5%": [results["R2"].quantile(0.025)],
        "R2 97.5%": [results["R2"].quantile(0.975)],

        "RMSE Mean": [results["RMSE"].mean()],
        "RMSE 2.5%": [results["RMSE"].quantile(0.025)],
        "RMSE 97.5%": [results["RMSE"].quantile(0.975)],

        "MAE Mean": [results["MAE"].mean()],
        "MAE 2.5%": [results["MAE"].quantile(0.025)],
        "MAE 97.5%": [results["MAE"].quantile(0.975)]
    })

    return ci_summary.round(3)


rf_ci = bootstrap_confidence_intervals(rf_boot_results, "Tuned Random Forest")
xgb_ci = bootstrap_confidence_intervals(xgb_boot_results, "Tuned XGBoost")

boot_ci_table = pd.concat([rf_ci, xgb_ci], ignore_index=True)
boot_ci_table

In [ ]:
boot_compare = pd.DataFrame({
    "Model": ["Tuned Random Forest", "Tuned XGBoost"],
    "Mean R2": [rf_boot_results["R2"].mean(), xgb_boot_results["R2"].mean()],
    "Mean RMSE": [rf_boot_results["RMSE"].mean(), xgb_boot_results["RMSE"].mean()],
    "Mean MAE": [rf_boot_results["MAE"].mean(), xgb_boot_results["MAE"].mean()]
}).round(3)

boot_compare = boot_compare.sort_values(by="Mean RMSE").reset_index(drop=True)
boot_compare

Distribution Plots of R²

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(rf_boot_results["R2"], bins=30, kde=True, edgecolor="black")
plt.title("Bootstrap Distribution of R² - Tuned Random Forest")
plt.xlabel("R²")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(xgb_boot_results["R2"], bins=30, kde=True, edgecolor="black")
plt.title("Bootstrap Distribution of R² - Tuned XGBoost")
plt.xlabel("R²")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

Distribution Plots of RMSE

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(rf_boot_results["RMSE"], bins=30, kde=True, edgecolor="black")
plt.title("Bootstrap Distribution of RMSE - Tuned Random Forest")
plt.xlabel("RMSE")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.histplot(xgb_boot_results["RMSE"], bins=30, kde=True, edgecolor="black")
plt.title("Bootstrap Distribution of RMSE - Tuned XGBoost")
plt.xlabel("RMSE")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
import os

plot_dir = r"D:\Thesis_Speed_Model\outputs\Plots"
table_dir = r"D:\Thesis_Speed_Model\outputs\Tables"

os.makedirs(plot_dir, exist_ok=True)
os.makedirs(table_dir, exist_ok=True)

print("Folders ready!")

In [ ]:
import os
import pandas as pd

table_dir = r"D:\Thesis_Speed_Model\outputs\Tables"
os.makedirs(table_dir, exist_ok=True)

excel_path = os.path.join(table_dir, "All_Model_Tables.xlsx")

# 🔷 Basic Tables
head_table = df.head().reset_index(drop=True)

descriptive_stats = df.describe().reset_index()
descriptive_stats.rename(columns={"index": "Statistic"}, inplace=True)

shape_table = pd.DataFrame({
    "Item": ["Rows", "Columns"],
    "Value": [df.shape[0], df.shape[1]]
})

missing_values_table = df.isnull().sum().reset_index()
missing_values_table.columns = ["Variable", "Missing Values"]

data_types_table = df.dtypes.reset_index()
data_types_table.columns = ["Variable", "Data Type"]

# 🔷 Correlation Matrix Table
corr_matrix_table = corr_matrix.reset_index()
corr_matrix_table.rename(columns={"index": "Variable"}, inplace=True)

# 🔷 Write everything into one Excel file
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    # 📌 Data Inspection
    head_table.to_excel(writer, sheet_name="First_5_Rows", index=False)
    descriptive_stats.to_excel(writer, sheet_name="Descriptive_Statistics", index=False)
    shape_table.to_excel(writer, sheet_name="Dataset_Shape", index=False)
    missing_values_table.to_excel(writer, sheet_name="Missing_Values", index=False)
    data_types_table.to_excel(writer, sheet_name="Data_Types", index=False)

    # 📌 EDA / Preparation
    skew_table.to_excel(writer, sheet_name="Skewness", index=False)
    corr_matrix_table.to_excel(writer, sheet_name="Correlation_Matrix", index=False)
    vif_data.to_excel(writer, sheet_name="VIF_Table", index=False)

    # 📌 Model Results
    mlr_final.to_excel(writer, sheet_name="MLR_Final", index=False)
    coef_table_mlr.to_excel(writer, sheet_name="MLR_Coefficients", index=False)

    rf_final.to_excel(writer, sheet_name="RF_Base", index=False)
    rf_tuned_final.to_excel(writer, sheet_name="RF_Tuned", index=False)

    xgb_final.to_excel(writer, sheet_name="XGB_Base", index=False)
    xgb_tuned_final.to_excel(writer, sheet_name="XGB_Tuned", index=False)

    svr_final.to_excel(writer, sheet_name="SVR_Base", index=False)
    svr_tuned_final.to_excel(writer, sheet_name="SVR_Tuned", index=False)

    ann_final.to_excel(writer, sheet_name="ANN_Base", index=False)
    ann_tuned_final.to_excel(writer, sheet_name="ANN_Tuned", index=False)

    # 📌 Comparison Tables
    rf_comparison.to_excel(writer, sheet_name="RF_Comparison", index=False)
    xgb_comparison.to_excel(writer, sheet_name="XGB_Comparison", index=False)
    svr_comparison.to_excel(writer, sheet_name="SVR_Comparison", index=False)
    ann_comparison.to_excel(writer, sheet_name="ANN_Comparison", index=False)
    master_table.to_excel(writer, sheet_name="Master_Table", index=False)

    # 📌 Best Hyperparameters
    best_rf_params.to_excel(writer, sheet_name="RF_Params", index=False)
    best_xgb_params.to_excel(writer, sheet_name="XGB_Params", index=False)
    best_svr_params.to_excel(writer, sheet_name="SVR_Params", index=False)
    best_ann_params.to_excel(writer, sheet_name="ANN_Params", index=False)

    # 📌 Feature Importance
    rf_tuned_importance.to_excel(writer, sheet_name="RF_Importance", index=False)
    xgb_tuned_importance.to_excel(writer, sheet_name="XGB_Importance", index=False)

    # 📌 Bootstrap Results (if available)
    if "boot_summary" in globals():
        boot_summary.to_excel(writer, sheet_name="Bootstrap_Summary", index=False)

    if "boot_ci_table" in globals():
        boot_ci_table.to_excel(writer, sheet_name="Bootstrap_CI", index=False)

print(f"✅ Excel file saved at:\n{excel_path}")

In [ ]:
import os
import matplotlib.pyplot as plt

plot_dir = r"D:\Thesis_Speed_Model\outputs\Plots"
os.makedirs(plot_dir, exist_ok=True)


def save_plot_from_title():
    # Get current plot title
    title = plt.gca().get_title()

    # Clean title to make valid filename
    filename = title.replace(":", "").replace(" ", "_").replace("/", "_") + ".png"

    filepath = os.path.join(plot_dir, filename)

    plt.tight_layout()
    plt.savefig(filepath, dpi=300, bbox_inches="tight")

    print(f"Saved: {filepath}")

In [ ]:
# ============================================================
# PARTIAL DEPENDENCE PLOTS (PDP)
# Using UN-SCALED data for better interpretation
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import PartialDependenceDisplay

# ------------------------------------------------------------
# 1. Extract best parameters from tuned RF
# ------------------------------------------------------------

best_params = grid_rf.best_params_

print("Best RF Parameters:", best_params)

# ------------------------------------------------------------
# 2. Retrain RF on ORIGINAL (unscaled) data
#    (Tree models do NOT require scaling)
# ------------------------------------------------------------

rf_unscaled = RandomForestRegressor(
    n_estimators=best_params["n_estimators"],
    max_depth=best_params["max_depth"],
    min_samples_split=best_params["min_samples_split"],
    min_samples_leaf=best_params["min_samples_leaf"],
    max_features=best_params["max_features"],
    random_state=42
)

rf_unscaled.fit(X_train, y_train)

# ------------------------------------------------------------
# 3. Check column names (IMPORTANT)
# ------------------------------------------------------------

print("Columns:", X_train.columns)

# ------------------------------------------------------------
# 4. PDP for Gradient
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 4))

PartialDependenceDisplay.from_estimator(
    rf_unscaled,
    X_train,
    features=["Gradient"],
    grid_resolution=50,
    ax=ax
)

ax.set_title("Partial Dependence Plot: Gradient vs V85", fontsize=12, fontweight="bold")
ax.set_xlabel("Gradient (%)")
ax.set_ylabel("Predicted V85 Speed (km/h)")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 5. PDP for Radius of Curve
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 4))

PartialDependenceDisplay.from_estimator(
    rf_unscaled,
    X_train,
    features=["Radius of Curve"],
    grid_resolution=50,
    ax=ax
)

ax.set_title("Partial Dependence Plot: Radius vs V85", fontsize=12, fontweight="bold")
ax.set_xlabel("Radius of Curve (m)")
ax.set_ylabel("Predicted V85 Speed (km/h)")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 6. BOTH PDP side by side (BEST FOR SLIDE)
# ------------------------------------------------------------

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

PartialDependenceDisplay.from_estimator(
    rf_unscaled,
    X_train,
    features=["Gradient"],
    grid_resolution=50,
    ax=ax[0]
)

PartialDependenceDisplay.from_estimator(
    rf_unscaled,
    X_train,
    features=["Radius of Curve"],
    grid_resolution=50,
    ax=ax[1]
)

ax[0].set_title("Gradient Effect on V85", fontsize=11, fontweight="bold")
ax[0].set_xlabel("Gradient (%)")
ax[0].set_ylabel("Predicted V85 Speed (km/h)")

ax[1].set_title("Radius Effect on V85", fontsize=11, fontweight="bold")
ax[1].set_xlabel("Radius of Curve (m)")
ax[1].set_ylabel("Predicted V85 Speed (km/h)")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 7. 2D PDP (INTERACTION — VERY STRONG FOR VIVA)
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 5))

PartialDependenceDisplay.from_estimator(
    rf_unscaled,
    X_train,
    features=[("Gradient", "Radius of Curve")],
    grid_resolution=30,
    ax=ax
)

ax.set_title("2D PDP: Gradient × Radius Interaction", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 3D PARTIAL DEPENDENCE PLOT
# Gradient × Radius
# Compatible with newer sklearn versions
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.inspection import partial_dependence

# Features
features = ["Gradient", "Radius of Curve"]

# Compute PDP
pdp_result = partial_dependence(
    rf_unscaled,
    X_train,
    features=features,
    grid_resolution=30
)

# Extract grid values
grad_vals = pdp_result["grid_values"][0]
radius_vals = pdp_result["grid_values"][1]

# Extract predicted partial dependence values
Z = pdp_result["average"][0]

# Create meshgrid
X_grid, Y_grid = np.meshgrid(grad_vals, radius_vals)

# IMPORTANT:
# If shape mismatch occurs, transpose Z
if Z.shape != X_grid.shape:
    Z = Z.T

# Plot 3D surface
fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection="3d")

surf = ax.plot_surface(
    X_grid,
    Y_grid,
    Z,
    cmap="viridis",
    edgecolor="none",
    alpha=0.9
)

ax.set_xlabel("Gradient (%)")
ax.set_ylabel("Radius of Curve (m)")
ax.set_zlabel("Predicted V85 Speed (km/h)")
ax.set_title("3D Partial Dependence: Gradient × Radius", fontsize=12, fontweight="bold")

fig.colorbar(surf, shrink=0.5, aspect=8, label="Predicted V85 Speed (km/h)")

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# V85 SPEED PREDICTION TOOL
# Tuned Random Forest Model — Hilly Roads of Nepal
# Add this code at the END of your final_model_analysis.ipynb
# =============================================================================

import joblib
import numpy as np
import pandas as pd

# =============================================================================
# STEP 1: SAVE THE TRAINED MODEL AND SCALER
# Run this once after your model training is complete
# =============================================================================

# Save the best tuned Random Forest model
joblib.dump(best_rf, 'best_rf_v85_model.pkl')

# IMPORTANT: Save the scaler too — you MUST use the same scaler for prediction
# Your notebook fits the scaler on X_train, so we save that exact scaler
joblib.dump(scaler, 'v85_scaler.pkl')

print("Model saved as: best_rf_v85_model.pkl")
print("Scaler saved as: v85_scaler.pkl")

# Also save feature column names to avoid input order mistakes
feature_columns = list(X.columns)
joblib.dump(feature_columns, 'v85_feature_columns.pkl')
print("Feature columns saved as: v85_feature_columns.pkl")
print("\nFeature order is:")
for i, col in enumerate(feature_columns):
    print(f"  {i+1}. {col}")


# =============================================================================
# STEP 2: LOAD AND PREDICT (use this anytime, even in a new session)
# =============================================================================

# Load saved model, scaler, and feature columns
loaded_model   = joblib.load('best_rf_v85_model.pkl')
loaded_scaler  = joblib.load('v85_scaler.pkl')
loaded_columns = joblib.load('v85_feature_columns.pkl')

print("Model, scaler, and feature columns loaded successfully.")
print(f"Expected features: {loaded_columns}")


# =============================================================================
# STEP 3: PREDICTION FUNCTION
# Matches your exact feature set:
#   Gradient, Radius of Curve, Carriageway Width,
#   Left Shoulder Width, Right Shoulder Width,
#   Left Shoulder Type_Gravel, Left Shoulder Type_Paved,
#   Right Shoulder Type_Gravel, Right Shoulder Type_Paved
# =============================================================================

def predict_v85(
    gradient,           # Longitudinal gradient in %  (e.g. 3.5)
    radius,             # Horizontal curve radius in m (e.g. 300). Use 1500 for tangent sections.
    carriageway_width,  # Carriageway width in m       (e.g. 6.0)
    left_shoulder_width,    # Left shoulder width in m    (e.g. 0.5)
    right_shoulder_width,   # Right shoulder width in m   (e.g. 0.5)
    left_shoulder_type,     # 'Absent', 'Gravel', or 'Paved'
    right_shoulder_type     # 'Absent', 'Gravel', or 'Paved'
):
    """
    Predicts V85 operating speed (km/h) for a hilly road section in Nepal.

    Shoulder type encoding (matches your get_dummies with drop_first=True):
        Absent  → Left_Gravel=0, Left_Paved=0
        Gravel  → Left_Gravel=1, Left_Paved=0
        Paved   → Left_Gravel=0, Left_Paved=1
    Same logic applies to Right Shoulder Type.

    Returns: predicted V85 in km/h
    """

    # --- Encode left shoulder type ---
    left_shoulder_type = left_shoulder_type.strip().capitalize()
    if left_shoulder_type == 'Absent':
        left_gravel = 0
        left_paved  = 0
    elif left_shoulder_type == 'Gravel':
        left_gravel = 1
        left_paved  = 0
    elif left_shoulder_type == 'Paved':
        left_gravel = 0
        left_paved  = 1
    else:
        raise ValueError(f"Invalid left_shoulder_type: '{left_shoulder_type}'. Use 'Absent', 'Gravel', or 'Paved'.")

    # --- Encode right shoulder type ---
    right_shoulder_type = right_shoulder_type.strip().capitalize()
    if right_shoulder_type == 'Absent':
        right_gravel = 0
        right_paved  = 0
    elif right_shoulder_type == 'Gravel':
        right_gravel = 1
        right_paved  = 0
    elif right_shoulder_type == 'Paved':
        right_gravel = 0
        right_paved  = 1
    else:
        raise ValueError(f"Invalid right_shoulder_type: '{right_shoulder_type}'. Use 'Absent', 'Gravel', or 'Paved'.")

    # --- Build input in correct feature order ---
    # Order must match: X.columns from your notebook
    # ['Gradient', 'Radius of Curve', 'Carriageway Width',
    #  'Left Shoulder Width', 'Right Shoulder Width',
    #  'Left Shoulder Type_Gravel', 'Left Shoulder Type_Paved',
    #  'Right Shoulder Type_Gravel', 'Right Shoulder Type_Paved']

    input_data = pd.DataFrame([{
        'Gradient'                  : gradient,
        'Radius of Curve'           : radius,
        'Carriageway Width'         : carriageway_width,
        'Left Shoulder Width'       : left_shoulder_width,
        'Right Shoulder Width'      : right_shoulder_width,
        'Left Shoulder Type_Gravel' : left_gravel,
        'Left Shoulder Type_Paved'  : left_paved,
        'Right Shoulder Type_Gravel': right_gravel,
        'Right Shoulder Type_Paved' : right_paved
    }])

    # Ensure column order exactly matches training
    input_data = input_data[loaded_columns]

    # --- Scale the input (same scaler fitted on X_train) ---
    input_scaled = loaded_scaler.transform(input_data)

    # --- Predict ---
    predicted_speed = loaded_model.predict(input_scaled)[0]

    return round(predicted_speed, 2)


# =============================================================================
# STEP 4: EXAMPLE PREDICTIONS — Test with real-world site conditions
# =============================================================================

print("\n" + "="*60)
print("EXAMPLE PREDICTIONS")
print("="*60)

# Example 1: Sharp curve, steep gradient (typical Nagdhunga-Mugling)
v85_1 = predict_v85(
    gradient=6.0,
    radius=50,
    carriageway_width=5.5,
    left_shoulder_width=0.3,
    right_shoulder_width=0.3,
    left_shoulder_type='Absent',
    right_shoulder_type='Absent'
)
print(f"\nExample 1 — Sharp curve (R=50m), steep gradient (6%):")
print(f"  Predicted V85 = {v85_1} km/h")

# Example 2: Moderate curve, moderate gradient
v85_2 = predict_v85(
    gradient=4.0,
    radius=300,
    carriageway_width=6.0,
    left_shoulder_width=0.5,
    right_shoulder_width=0.5,
    left_shoulder_type='Gravel',
    right_shoulder_type='Gravel'
)
print(f"\nExample 2 — Moderate curve (R=300m), moderate gradient (4%):")
print(f"  Predicted V85 = {v85_2} km/h")

# Example 3: Tangent section (R=1500m), gentle gradient
v85_3 = predict_v85(
    gradient=2.0,
    radius=1500,
    carriageway_width=7.0,
    left_shoulder_width=0.8,
    right_shoulder_width=0.8,
    left_shoulder_type='Paved',
    right_shoulder_type='Paved'
)
print(f"\nExample 3 — Tangent section (R=1500m), gentle gradient (2%):")
print(f"  Predicted V85 = {v85_3} km/h")

# Example 4: Your own site — just change the values
v85_custom = predict_v85(
    gradient=3.5,       # % gradient
    radius=200,         # curve radius in meters
    carriageway_width=6.0,
    left_shoulder_width=0.5,
    right_shoulder_width=0.5,
    left_shoulder_type='Gravel',
    right_shoulder_type='Absent'
)
print(f"\nExample 4 — Custom site (R=200m, G=3.5%, CW=6.0m):")
print(f"  Predicted V85 = {v85_custom} km/h")


# =============================================================================
# STEP 5: BATCH PREDICTION — Predict for multiple sites at once
# =============================================================================

print("\n" + "="*60)
print("BATCH PREDICTION — Multiple Sites")
print("="*60)

# Create a DataFrame of new sites (e.g., proposed road alignment)
new_sites = pd.DataFrame({
    'Gradient'            : [3.0, 5.0, 2.0, 7.0, 4.5],
    'Radius of Curve'     : [100, 250, 1500, 75, 500],
    'Carriageway Width'   : [5.5, 6.0, 7.0, 5.0, 6.5],
    'Left Shoulder Width' : [0.3, 0.5, 0.8, 0.0, 0.6],
    'Right Shoulder Width': [0.3, 0.5, 0.8, 0.0, 0.6],
    'Left Shoulder Type'  : ['Absent', 'Gravel', 'Paved', 'Absent', 'Gravel'],
    'Right Shoulder Type' : ['Absent', 'Gravel', 'Paved', 'Absent', 'Gravel']
})

# Predict for each site
new_sites['Predicted V85 (km/h)'] = new_sites.apply(
    lambda row: predict_v85(
        gradient=row['Gradient'],
        radius=row['Radius of Curve'],
        carriageway_width=row['Carriageway Width'],
        left_shoulder_width=row['Left Shoulder Width'],
        right_shoulder_width=row['Right Shoulder Width'],
        left_shoulder_type=row['Left Shoulder Type'],
        right_shoulder_type=row['Right Shoulder Type']
    ),
    axis=1
)

print("\nBatch Prediction Results:")
print(new_sites[['Gradient', 'Radius of Curve', 'Carriageway Width',
                  'Left Shoulder Type', 'Right Shoulder Type',
                  'Predicted V85 (km/h)']].to_string(index=False))


# =============================================================================
# STEP 6: PARTIAL DEPENDENCE — How does V85 change with Radius?
# (Visual tool equivalent to MLR equation — for your thesis)
# =============================================================================

print("\n" + "="*60)
print("PARTIAL DEPENDENCE — V85 vs Radius of Curve")
print("(All other variables held at mean values)")
print("="*60)

# Mean values of other variables from your dataset
mean_gradient        = df['Gradient'].mean()
mean_cw_width        = df['Carriageway Width'].mean()
mean_ls_width        = df['Left Shoulder Width'].mean()
mean_rs_width        = df['Right Shoulder Width'].mean()

# Most common shoulder type
most_common_ls = df['Left Shoulder Type'].mode()[0]
most_common_rs = df['Right Shoulder Type'].mode()[0]

# Range of radius values to test
radius_range = [15, 30, 50, 75, 100, 150, 200, 300, 400, 500,
                600, 700, 800, 1000, 1200, 1500]

pdp_results = []
for r in radius_range:
    v85_pred = predict_v85(
        gradient=mean_gradient,
        radius=r,
        carriageway_width=mean_cw_width,
        left_shoulder_width=mean_ls_width,
        right_shoulder_width=mean_rs_width,
        left_shoulder_type=most_common_ls,
        right_shoulder_type=most_common_rs
    )
    pdp_results.append({'Radius (m)': r, 'Predicted V85 (km/h)': v85_pred})

pdp_df = pd.DataFrame(pdp_results)
print(pdp_df.to_string(index=False))

# Plot the Partial Dependence for Radius
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.figure(figsize=(8, 4))
plt.plot(pdp_df['Radius (m)'], pdp_df['Predicted V85 (km/h)'],
         marker='o', color='steelblue', linewidth=2, markersize=5)
plt.xlabel('Radius of Curve (m)', fontsize=12)
plt.ylabel('Predicted V85 Speed (km/h)', fontsize=12)
plt.title('Partial Dependence Plot — Radius of Curve vs Predicted V85\n'
          '(Other variables held at mean)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('PDP_Radius_vs_V85.png', dpi=300)
plt.show()
print("\nPDP plot saved as: PDP_Radius_vs_V85.png")


# =============================================================================
# STEP 7: PARTIAL DEPENDENCE — V85 vs Gradient
# This directly addresses your supervisor's comment about gradient's
# near-zero r but 21% RF importance
# =============================================================================

gradient_range = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0,
                  4.5, 5.0, 5.5, 6.0, 7.0, 8.0, 9.0, 10.0]

mean_radius = df['Radius of Curve'].mean()

pdp_gradient = []
for g in gradient_range:
    v85_pred = predict_v85(
        gradient=g,
        radius=mean_radius,
        carriageway_width=mean_cw_width,
        left_shoulder_width=mean_ls_width,
        right_shoulder_width=mean_rs_width,
        left_shoulder_type=most_common_ls,
        right_shoulder_type=most_common_rs
    )
    pdp_gradient.append({'Gradient (%)': g, 'Predicted V85 (km/h)': v85_pred})

pdp_gradient_df = pd.DataFrame(pdp_gradient)
print("\nPartial Dependence — Gradient vs V85:")
print(pdp_gradient_df.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.plot(pdp_gradient_df['Gradient (%)'], pdp_gradient_df['Predicted V85 (km/h)'],
         marker='o', color='darkorange', linewidth=2, markersize=5)
plt.xlabel('Gradient (%)', fontsize=12)
plt.ylabel('Predicted V85 Speed (km/h)', fontsize=12)
plt.title('Partial Dependence Plot — Gradient vs Predicted V85\n'
          '(Other variables held at mean)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('PDP_Gradient_vs_V85.png', dpi=300)
plt.show()
print("PDP plot saved as: PDP_Gradient_vs_V85.png")
print("\nNOTE: This plot demonstrates the NONLINEAR effect of gradient")
print("that Pearson r cannot detect — use this to defend the 21% RF importance finding.")


# =============================================================================
# STEP 8: DESIGN LOOKUP TABLE
# Practical tool for road engineers — equivalent to MLR equation
# =============================================================================

print("\n" + "="*60)
print("DESIGN LOOKUP TABLE — Predicted V85 (km/h)")
print("Gradient vs Radius of Curve (CW=6.0m, Shoulders=Gravel, 0.5m)")
print("="*60)

radius_vals   = [50, 100, 200, 300, 500, 700, 1000, 1500]
gradient_vals = [2, 4, 6, 8, 10]

lookup_rows = []
for g in gradient_vals:
    row = {'Gradient (%)': g}
    for r in radius_vals:
        v85 = predict_v85(
            gradient=g,
            radius=r,
            carriageway_width=6.0,
            left_shoulder_width=0.5,
            right_shoulder_width=0.5,
            left_shoulder_type='Gravel',
            right_shoulder_type='Gravel'
        )
        row[f'R={r}m'] = v85
    lookup_rows.append(row)

lookup_table = pd.DataFrame(lookup_rows)
print(lookup_table.to_string(index=False))

# Save to Excel for practical use
lookup_table.to_excel('V85_Design_Lookup_Table.xlsx', index=False)
print("\nLookup table saved as: V85_Design_Lookup_Table.xlsx")
print("Engineers can use this table directly for design consistency evaluation.")